In [1]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

device= 'cuda' if torch.cuda.is_available() else 'cpu'

seed= 42
torch.random.manual_seed(seed)

root= 'pictures/'

In [2]:
train_df= pd.read_csv("train_data.csv")
test_df= pd.read_csv("test_data.csv")

unique_sizes= []

for _, row in train_df.iterrows():
    unique_sizes.append(Image.open(root+ 'train/'+ str(row['ID'])+ '.jpg').size)

print(np.unique(unique_sizes, axis= 0))

idx_to_label=[label for label in train_df['label'].unique()]
label_to_idx={label: idx for idx, label in zip(range(len(idx_to_label)), idx_to_label)}

[[256 256]]


In [3]:
from torch.utils.data import Dataset, DataLoader

class ImageDataset(Dataset):
    def __init__(self, df, transform= None):
        self.df= df
        self.transform= transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row= self.df.iloc[idx]
        if 'label' in row:
            img_path= root+ 'train/'+ str(row['ID']) + ".jpg"
            img= Image.open(img_path)

            if self.transform:
                img= self.transform(img)
            
            label= torch.tensor(label_to_idx[row['label']], dtype= torch.long)

            return img, label
        else:
            img_path= root+ 'test/'+ str(row['ID']) + ".jpg"

            img= Image.open(img_path)

            if self.transform:
                img= self.transform(img)

            return img

In [4]:
from torchvision import transforms

train_transforms= transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transforms= transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [5]:
train_dataset= ImageDataset(train_df, transform=train_transforms)
test_dataset= ImageDataset(test_df, transform=test_transforms)

train_dataloader= DataLoader(train_dataset, batch_size=32, shuffle= True)
test_dataloader= DataLoader(test_dataset, batch_size=32, shuffle= False)

In [6]:
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn

model= resnet50(weights= ResNet50_Weights.IMAGENET1K_V2)
model.fc= nn.Linear(model.fc.in_features, 6)

In [7]:
epochs= 5
lr= 1e-4

model.to(device)

criterion= nn.CrossEntropyLoss()
optimizer= torch.optim.AdamW(model.parameters(), lr= lr)

for epoch in range(epochs):
    model.train()
    running_loss= 0.0
    for images, labels in train_dataloader:
        images, labels= images.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs= model(images)
        loss= criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss+= loss
    running_loss/= len(train_dataloader)
    print(f"Epoch: {epoch+ 1}, Loss: {running_loss}")

Epoch: 1, Loss: 0.5512072443962097
Epoch: 2, Loss: 0.19690360128879547
Epoch: 3, Loss: 0.12048573791980743
Epoch: 4, Loss: 0.08879722654819489
Epoch: 5, Loss: 0.07321224361658096


In [8]:
predictions= []

model.eval()

with torch.no_grad():
    for images in test_dataloader:
        images= images.to(device)

        outputs= model(images)
        preds= torch.argmax(outputs, dim= 1)
        predictions.extend(preds.cpu().numpy())

answer= []

for idx, row in test_df.iterrows():
    answer.append({
        'subtaskID': row['subtaskID'],
        'datapointID': row['ID'],
        'answer': idx_to_label[predictions[idx]]
    })

pd.DataFrame(answer).to_csv("submission.csv", index= False)